# Working with eye-tracking corpora

In this notebook, you will learn how to access, process, visualize, and analyze eye-tracking data with the Python package [pymovements](https://pymovements.readthedocs.io/).

In [ ]:
import pymovements as pm

## Downloading a dataset

pymovements has a [library of openly accessible datasets](https://pymovements.readthedocs.io/en/stable/datasets/index.html#public-datasets) that can be easily downloaded directly in Python code.

For this example, we'll download the GGTG dataset ([gaze-guided text generation](https://doi.org/10.18653/v1/2026.eacl-long.107)), which contains data from 24 participants reading English machine-generated texts that span multiple screens:

In [ ]:
ggtg = pm.Dataset("ggtg/ggtg.yaml", path="ggtg").download()

# To download a different dataset from the library, replace `ggtg/ggtg.yaml` with the name of the dataset:
# provo = pm.Dataset("Provo", path="provo").download(resume=True)

We can then load it into memory and inspect its contents:

In [ ]:
ggtg.load()

The eye-tracking data is stored in the `gaze` attribute of the dataset. In the GGTG dataset, you can find one `Gaze` object per participant. These contain the raw samples (i.e., pixel coordinates):

In [ ]:
ggtg.gaze[0].samples

## Visualizing samples

Since each participant read multiple screens of text, it is useful to split every `Gaze` object into multiple `Gaze` objects, each containing data for a single screen:

In [ ]:
ggtg.split_gaze_data(by="stimulus")
ggtg.gaze[0].samples

Now we can plot the first participant's data for the first screen:

In [ ]:
pm.plotting.traceplot(ggtg.gaze[0])

## Detecting and visualizing fixations

To detect fixations, we can use an algorithm like [I-DT](https://doi.org/10.1145/355017.355028). For this, we first need to convert the coordinates from pixels to degrees of visual angle (dva):

In [ ]:
ggtg.pix2deg()
ggtg.detect("idt", clear=True)
ggtg.events[0].frame

Now that we know where each fixation starts and ends, we can calculate the fixation location. To do this, we simply take the mean of the pixel coordinates of the samples that belong to a fixation:

In [ ]:
ggtg.compute_event_properties(("location", {"position_column": "pixel", "method": "mean"}))
ggtg.events[0].frame

Now we can plot the detected fixations:

In [ ]:
pm.plotting.scanpathplot(gaze=ggtg.gaze[0])

## Working with reading measures

Conveniently, the GGTG dataset already comes with several pre-computed reading measures. Again, we have one DataFrame per participant:

In [ ]:
ggtg.precomputed_reading_measures[0].frame

pymovements uses [polars](https://pola.rs/) DataFrames, which offer faster performance than [pandas](https://pandas.pydata.org/). If you prefer working with pandas, you can convert the data like this:

In [ ]:
reading_measures = ggtg.precomputed_reading_measures[0].frame.to_pandas()
reading_measures

Let's combine the reading measures from all participants into a single DataFrame and add the participant IDs:

In [ ]:
import pandas as pd

participant_ids = [f"P{str(i).zfill(2)}" for i in range(1, 25)]
all_reading_measures = pd.DataFrame()
for participant_id, reading_measures in zip(participant_ids, ggtg.precomputed_reading_measures):
    reading_measures = reading_measures.frame.to_pandas()
    reading_measures["participant_id"] = participant_id
    all_reading_measures = pd.concat([all_reading_measures, reading_measures], ignore_index=True)
all_reading_measures

### Task

The following code adds word length and frequency to the DataFrames. Your task is to **analyze the relationship between word length / word frequency and various reading measures**. You can do this by plotting or calculating the correlation.

- Which reading measures increase as words become longer or more frequent? Which decrease?
- Which reading measures are insensitive to changes in word length or frequency?
- Does this match your expectations?

In [ ]:
import wordfreq

all_reading_measures["word_length"] = all_reading_measures["aoi_content"].apply(len)
all_reading_measures["word_frequency"] = all_reading_measures["aoi_content"].apply(wordfreq.zipf_frequency, lang="en")
all_reading_measures

In [ ]:
# TODO: Your code here